# Results postprocessing

In [83]:
import pandas as pd

### Joining folders

In [84]:
import os

for folder in os.listdir('../'):
    if folder.startswith('results_'):
        folder_results_name = os.path.join('../', folder)
        sufix = '_'.join(folder.split('_')[1:])

        qtd_folders = sum(1 for x in os.listdir(folder_results_name) if os.path.isdir(os.path.join(folder_results_name, x)))
        if qtd_folders == 7:
            print('Folder {} already has been processed, skipping...'.format(folder_results_name))
            continue
        
        folders = os.listdir(folder_results_name).copy()
        for folder_model in folders:
            parts = folder_model.split('_')
            new_folder_name = parts[2] + '_' + sufix
            
            if not os.path.exists(os.path.join(folder_results_name, new_folder_name)):
                os.mkdir(os.path.join(folder_results_name, new_folder_name))

            os.rename(os.path.join(folder_results_name, folder_model), os.path.join(folder_results_name, new_folder_name, folder_model))

Folder ../results_triage_dermatoscope already has been processed, skipping...
Folder ../results_triage_clinical already has been processed, skipping...
Folder ../results_diag_clinical already has been processed, skipping...


### Getting metrics

In [85]:
df = pd.DataFrame(columns=["backbone", "classifier", "image_type", "folder", "metric", "metric_value"])

In [86]:
def getMetrics(path):
    with open(path, "r") as f:
        lines = f.read().split("\n")
        acc = float(lines[3].split(": ")[1])
        bacc = float(lines[5].split(": ")[1])

        auc_line = len(lines)
        if 'AUC' in lines[auc_line-1]:
            auc_line = auc_line-1
        elif 'AUC' in lines[auc_line-2]:
            auc_line = auc_line-2
        
        auc = float(lines[auc_line].split(": ")[1])
        
        for line in lines[22:]:
            if "macro avg" in line:
                f1_macro = float([x for x in line.split(" ") if x.strip() != ""][-2])
    
        return [('acc', acc), ('bacc', bacc), ('f1_macro', f1_macro), ('auc', auc)]

In [87]:
base_path = '../'
for file in os.listdir(base_path):
    if os.path.isdir(os.path.join(base_path, file)) and file.startswith("results_"):
        path = os.path.join(base_path, file)
        parts = file.split("_")
        classifier = parts[1]
        image_type = parts[2]
        
        for subfile in os.listdir(path):
            path2 = os.path.join(path, subfile)
            if os.path.isdir(path2):
                parts = subfile.split("_")
                backbone = parts[0]

                for folder in os.listdir(path2):
                    path3 = os.path.join(path2, folder)
                    if os.path.isdir(path3):
                        folder_number = folder.split("_")[-2]
                        results_path = os.path.join(path3, "best_metrics")

                        results = getMetrics(os.path.join(results_path, "metrics.txt"))
                        for metric in results:
                            new_row = {
                                "backbone": backbone,
                                "classifier": classifier,
                                "image_type": image_type,
                                "folder": folder_number,
                                "metric": metric[0],
                                "metric_value": metric[1],
                            }
                            df.loc[len(df)] = new_row

In [88]:
df['backbone'] = df['backbone'].map({
    'resnet-50': 'ResNet-50',
    'mobilenet': 'MobileNet-v2',
    'efficientnet-b4': 'EfficientNet-b4',
    'caformer': 'Caformer-s18',
    'davit': 'Davit-tiny',
    'maxvit': 'Maxvit-tiny',
    'mvitv2': 'Mvit2-small'
})

### Postprocessing results

In [89]:
def gerar_tabelas_latex(df, output_path='tabelas_overleaf.txt'):
    # Função interna para montar a tabela de um classificador específico
    def montar_tabela(df_class, nome_classificador):
        # 1. Agrupar os dados para calcular a média e o desvio padrão entre os "folders"
        df_agg = df_class.groupby(['backbone', 'image_type', 'metric'])['metric_value'].agg(['mean', 'std']).reset_index()
        
        # 2. Formatar as métricas no padrão matemático "média \pm std"
        df_agg['valor_formatado'] = df_agg.apply(
            lambda x: f"${x['mean']:.2f} \\pm {x['std']:.2f}$" if pd.notna(x['std']) else f"${x['mean']:.2f}$", 
            axis=1
        )
        
        # 3. Pivotar o dataframe: linhas=backbone, colunas=[image_type, metric]
        df_pivot = df_agg.pivot(index='backbone', columns=['image_type', 'metric'], values='valor_formatado')
        
        # 4. Forçar a ordem exata solicitada (Clinical primeiro, Dermatoscope depois)
        metricas_ordem = ['bacc', 'acc', 'auc', 'f1_macro']
        ordem_colunas = [('clinical', m) for m in metricas_ordem] + [('dermatoscope', m) for m in metricas_ordem]
        
        # Filtrar para evitar erro caso alguma métrica esteja faltando nos dados
        colunas_validas = [col for col in ordem_colunas if col in df_pivot.columns]
        df_pivot = df_pivot[colunas_validas]
        
        # 5. Construir o código LaTeX linha por linha
        linhas_latex = [
            "\\begin{table}[htbp]",
            "    \\centering",
            f"    \\caption{{Resultados de avaliação do classificador de {nome_classificador.capitalize()}.}}",
            f"    \\label{{tab:resultados_{nome_classificador}}}",
            "    \\resizebox{\\textwidth}{!}{",
            "    \\begin{tabular}{l" + "c" * len(colunas_validas) + "}",
            "        \\toprule",
            "        \\multirow{2}{*}{\\textbf{Backbone}} & \\multicolumn{4}{c}{\\textbf{Clinical}} & \\multicolumn{4}{c}{\\textbf{Dermatoscope}} \\\\",
            "        \\cmidrule(lr){2-5} \\cmidrule(lr){6-9}"
        ]
        
        # Montar a segunda linha do cabeçalho (nomes das métricas)
        cabecalho_metricas = " & ".join([f"\\textbf{{{m[1].upper()}}}" for m in colunas_validas])
        linhas_latex.append(f"         & {cabecalho_metricas} \\\\")
        linhas_latex.append("        \\midrule")
        
        # Preencher as linhas com os dados
        for backbone, row in df_pivot.iterrows():
            valores = " & ".join([str(v) if pd.notna(v) else "-" for v in row.values])
            linhas_latex.append(f"        \\textbf{{{backbone}}} & {valores} \\\\")
            
        # Fechar a tabela
        linhas_latex.extend([
            "        \\bottomrule",
            "    \\end{tabular}",
            "    }",
            "\\end{table}"
        ])
        
        return "\n".join(linhas_latex)

    # Filtrar dados para Triage e Diag e gerar o código de cada um
    df_triage = df[df['classifier'] == 'triage']
    tabela_triage = montar_tabela(df_triage, 'triage') if not df_triage.empty else "% Sem dados disponíveis para triage"
    
    df_diag = df[df['classifier'] == 'diag']
    tabela_diag = montar_tabela(df_diag, 'diag') if not df_diag.empty else "% Sem dados disponíveis para diag"
    
    # Salvar no arquivo TXT
    conteudo_final = f"% --- TABELA: TRIAGE ---\n\n{tabela_triage}\n\n\n% --- TABELA: DIAG ---\n\n{tabela_diag}\n"
    
    with open(output_path, 'w') as f:
        f.write(conteudo_final)
        
    print(f"Arquivo salvo com sucesso em: {output_path}")

# Como executar:
# gerar_tabelas_latex(df, 'tabelas_overleaf.txt')

In [90]:
gerar_tabelas_latex(df)

Arquivo salvo com sucesso em: tabelas_overleaf.txt


## Validation

In [91]:
df_val = pd.DataFrame(columns=["backbone", "classifier", "image_type", "folder", "metric", "metric_value"])

In [93]:
base_path = '../'
for file in os.listdir(base_path):
    if os.path.isdir(os.path.join(base_path, file)) and file.startswith("results_"):
        path = os.path.join(base_path, file)
        parts = file.split("_")
        classifier = parts[1]
        image_type = parts[2]
        
        for subfile in os.listdir(path):
            path2 = os.path.join(path, subfile)
            if os.path.isdir(path2):
                parts = subfile.split("_")
                backbone = parts[0]

                for folder in os.listdir(path2):
                    path3 = os.path.join(path2, folder)
                    if os.path.isdir(path3):
                        folder_number = folder.split("_")[-2]
                        results_path = os.path.join(path3, "test_pred")
                        results = getMetrics(os.path.join(results_path, "metrics.txt"))
                        for metric in results:
                            new_row = {
                                "backbone": backbone,
                                "classifier": classifier,
                                "image_type": image_type,
                                "folder": folder_number,
                                "metric": metric[0],
                                "metric_value": metric[1],
                            }
                            df_val.loc[len(df_val)] = new_row

In [94]:
df_val['backbone'] = df_val['backbone'].map({
    'resnet-50': 'ResNet-50',
    'mobilenet': 'MobileNet-v2',
    'efficientnet-b4': 'EfficientNet-b4',
    'caformer': 'Caformer-s18',
    'davit': 'Davit-tiny',
    'maxvit': 'Maxvit-tiny',
    'mvitv2': 'Mvit2-small'
})

In [96]:
gerar_tabelas_latex(df_val, output_path='tabelas_val.txt')

Arquivo salvo com sucesso em: tabelas_val.txt
